In [33]:
import pandas as pd
from tabpfn import TabPFNClassifier, TabPFNRegressor
import numpy as np
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedShuffleSplit,ShuffleSplit
from sklearn.preprocessing import OneHotEncoder
load_dotenv(".env")

True

In [34]:
blood_dataset = pd.read_csv("datasets/blood_dataset/blood.csv")
print(blood_dataset.head())
data_size = len(blood_dataset)
print(data_size)

   Recency  Frequency  Monetary  Time  Class
0        2         50     12500    99      1
1        0         13      3250    28      1
2        1         17      4000    36      1
3        2         20      5000    45      1
4        1         24      6000    77      0
748


In [35]:
from sklearn.tree import DecisionTreeRegressor,DecisionTreeClassifier,plot_tree
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
from sklearn.neural_network import MLPClassifier,MLPRegressor

In [ ]:

from sklearn.metrics import accuracy_score
data = np.array(blood_dataset.drop(columns=["Class"]),dtype=np.float32)
targets = np.array(blood_dataset["Class"],dtype=np.float32)
def evaluate_dataset_classification(data,targets,n_splits=5,n_train_examples=32):
    sss = StratifiedShuffleSplit(n_splits=n_splits,train_size=n_train_examples)
    pfn_accuracies = []
    distilled_tree_accuracies = []
    tree_accuracies = []
    for i,(train_index, test_index) in enumerate(sss.split(data, targets)):
        print(i)
        X_train = data[train_index]
        y_train = targets[train_index]
        X_test = data[test_index]
        y_test = targets[test_index]
        clf = TabPFNClassifier()
        clf.fit(X_train, y_train)
        y_distil = clf.predict_proba(X_train)
        print(y_distil.shape)
        y_distil = y_distil[:,0]-y_distil[:,1]
        distilled_decision_tree = MLPRegressor(hidden_layer_sizes=(50,))#RandomForestRegressor()
        distilled_decision_tree.fit(X_train,y_distil)
        pfn_predictions = clf.predict(X_test)
        print("done")
        distilled_tree_predictions = (distilled_decision_tree.predict(X_test)<0).astype(np.int16)
        decision_tree = MLPClassifier(hidden_layer_sizes=(50,))#RandomForestClassifier()
        decision_tree.fit(X_train,y_train)
        tree_predictions = decision_tree.predict(X_test)
        #print("---------------")
        pfn_accuracies.append(accuracy_score(y_test,pfn_predictions))
        distilled_tree_accuracies.append(accuracy_score(y_test,distilled_tree_predictions))
        tree_accuracies.append(accuracy_score(y_test,tree_predictions))
        #print(f"PFN accuracy: {accuracy_score(y_test,pfn_predictions):3f}")
        #print(f"distillated tree accuracy: {accuracy_score(y_test,distilled_tree_predictions):3f}")
        #print(f"tree accuracy: {accuracy_score(y_test,tree_predictions):3f}")
    return np.mean(pfn_accuracies).item(),np.mean(distilled_tree_accuracies).item(),np.mean(tree_accuracies).item()
print(evaluate_dataset_classification(data,targets))

0
(32, 2)
done
1
(32, 2)
done
2
(32, 2)
done
3
(32, 2)
done
4
(32, 2)
done
(0.7628491620111733, 0.5519553072625698, 0.523463687150838)


In [43]:
bank_dataset = pd.read_csv("datasets/banking_dataset/train.csv",delimiter=";")#+pd.read_csv("datasets/banking_dataset/test.csv",delimiter=";")
onehotenc = OneHotEncoder()
categorical_features = ["job","marital","education","default","housing","loan","contact","month","poutcome"]
targets = bank_dataset["y"]
data = bank_dataset.drop(columns=["y"])
categorical = np.array(onehotenc.fit_transform(data[categorical_features]).todense())
non_categorical = np.array(data.drop(columns=categorical_features))
X = np.concat([categorical,non_categorical],axis=1)
y = np.array(targets=="yes",dtype=np.int16)
print(X.shape)
print(evaluate_dataset_classification(X[:5000],y[:5000],n_train_examples=32))

(45211, 51)
0
(32, 2)


/Users/filipzdebik/Documents/Programming/tabpfn-distill/TabPFN-Distil/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


done
1
(32, 2)
done
2
(32, 2)
done
3
(32, 2)
done
4
(32, 2)


/Users/filipzdebik/Documents/Programming/tabpfn-distill/TabPFN-Distil/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


done
(0.9694444444444443, 0.5490338164251207, 0.9482286634460546)


In [ ]:
from sklearn.metrics import mean_squared_error
def evaluate_dataset_regression(data,targets,n_splits=5,n_train_examples=32):
    sss = ShuffleSplit(n_splits=n_splits,train_size=n_train_examples/data_size)
    pfn_accuracies = []
    distilled_tree_accuracies = []
    tree_accuracies = []
    for i,(train_index, test_index) in enumerate(sss.split(data, targets)):
        print(i)
        X_train = data[train_index]
        y_train = targets[train_index]
        X_test = data[test_index]
        y_test = targets[test_index]
        clf = TabPFNRegressor()
        clf.fit(X_train, y_train)
        y_distil = clf.predict(X_train)
        distilled_decision_tree = MLPRegressor(hidden_layer_sizes=(50,))#RandomForestRegressor()
        distilled_decision_tree.fit(X_train,y_distil)
        pfn_predictions = clf.predict(X_test)
        print("done")
        distilled_tree_predictions = distilled_decision_tree.predict(X_test)
        decision_tree = MLPRegressor(hidden_layer_sizes=(50,))#RandomForestRegressor()
        decision_tree.fit(X_train,y_train)
        tree_predictions = decision_tree.predict(X_test)
        #print("---------------")
        pfn_accuracies.append(mean_squared_error(y_test,pfn_predictions))
        distilled_tree_accuracies.append(mean_squared_error(y_test,distilled_tree_predictions))
        tree_accuracies.append(mean_squared_error(y_test,tree_predictions))
        #print(f"PFN accuracy: {accuracy_score(y_test,pfn_predictions):3f}")
        #print(f"distillated tree accuracy: {accuracy_score(y_test,distilled_tree_predictions):3f}")
        #print(f"tree accuracy: {accuracy_score(y_test,tree_predictions):3f}")
    return np.mean(pfn_accuracies).item(),np.mean(distilled_tree_accuracies).item(),np.mean(tree_accuracies).item()

In [ ]:
california_housing = pd.read_csv("datasets/california_housing/housing.csv").dropna()
target = california_housing["median_house_value"]
data = california_housing.drop(columns=["median_house_value"])
categorical = np.array(onehotenc.fit_transform(data[["ocean_proximity"]]).todense(),dtype=np.float32)
non_categorical = np.array(data.drop(columns=["ocean_proximity"]),dtype=np.float32)
X = np.concat([categorical,non_categorical],axis=1)
y = np.array(target,dtype=np.float32)
print(X.shape)
evaluate_dataset_regression(X[:1000],y[:1000],n_splits=5,n_train_examples=32)

(20433, 13)
0


/Users/filipzdebik/Documents/Programming/tabpfn-distill/TabPFN-Distil/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


done
1


/Users/filipzdebik/Documents/Programming/tabpfn-distill/TabPFN-Distil/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/filipzdebik/Documents/Programming/tabpfn-distill/TabPFN-Distil/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


done
2


...

done
4


/Users/filipzdebik/Documents/Programming/tabpfn-distill/TabPFN-Distil/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/filipzdebik/Documents/Programming/tabpfn-distill/TabPFN-Distil/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


done


/Users/filipzdebik/Documents/Programming/tabpfn-distill/TabPFN-Distil/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


(2944521318.4, 48031057510.4, 46675274956.8)